In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Point-in-time requirement: historical aggregates must use only records available
# no later than the application decision time. Do not include post-decision data.
# The public Home Credit data lacks a reliable application timestamp, so downstream
# results use a stratified random holdout rather than claiming true time-out validation.

app_train = pd.read_csv('../data/raw/application_train.csv', encoding='cp1252')
app_test  = pd.read_csv('../data/raw/application_test.csv',  encoding='cp1252')
bureau    = pd.read_csv('../data/raw/bureau.csv',            encoding='cp1252')
bb        = pd.read_csv('../data/raw/bureau_balance.csv',    encoding='cp1252')
prev      = pd.read_csv('../data/raw/previous_application.csv', encoding='cp1252')
inst      = pd.read_csv('../data/raw/installments_payments.csv', encoding='cp1252')
cc        = pd.read_csv('../data/raw/credit_card_balance.csv',   encoding='cp1252')
pos       = pd.read_csv('../data/raw/POS_CASH_balance.csv',      encoding='cp1252')

print("Shapes:")
for name, df in zip(['app_train','bureau','bb','prev','inst','cc','pos'],
                    [app_train, bureau, bb, prev, inst, cc, pos]):
    print(f"  {name}: {df.shape}")

Shapes:
  app_train: (307511, 122)
  bureau: (1716428, 17)
  bb: (27299925, 3)
  prev: (1670214, 37)
  inst: (13605401, 8)
  cc: (3840312, 23)
  pos: (10001358, 8)


In [2]:
# 将征信逾期状态STATUS映射为代表逾期严重程度的数字
# C/X/0：无逾期，赋值0；1~5分别代表逾期30/60/90/120/150+天，数字越大逾期越严重
status_map = {'C': 0, 'X': 0, '0': 0, '1': 1, '2': 2, '3': 3, '4': 4, '5': 5}
bb['STATUS_NUM'] = bb['STATUS'].map(status_map)

# 按客户征信ID分组，聚合每个客户全历史信贷账户的逾期指标
bb_agg = bb.groupby('SK_ID_BUREAU').agg(
    # bb_months_count   = ('MONTHS_BALANCE', 'count'), # 该条信贷账户总记录月数
    bb_max_dpd        = ('STATUS_NUM', 'max'),   # 历史最严重逾期等级（最大值，代表最坏逾期记录）
    bb_mean_dpd       = ('STATUS_NUM', 'mean'), # 平均逾期严重程度
    bb_dpd_months     = ('STATUS_NUM', lambda x: (x > 0).sum()),  # 存在逾期的总月份数（STATUS_NUM>0即为有逾期）
).reset_index()

print(bb_agg.shape)
bb_agg.head()

(817395, 4)


,SK_ID_BUREAU,bb_max_dpd,bb_mean_dpd,bb_dpd_months
0,5001709,0,0.0,0
1,5001710,0,0.0,0
2,5001711,0,0.0,0
3,5001712,0,0.0,0
4,5001713,0,0.0,0


In [3]:
# left保留bureau全部信贷账户，匹配对应月度逾期统计指标
bureau_full = bureau.merge(bb_agg, on='SK_ID_BUREAU', how='left')

# 按客户唯一ID分组，聚合该客户名下所有信贷账户的全局征信特征
bureau_agg = bureau_full.groupby('SK_ID_CURR').agg(
    # ========== 1. 信贷账户数量类特征 ==========
    bureau_loan_count      = ('SK_ID_BUREAU', 'count'),               # 客户总信贷账户条数
    bureau_active_count    = ('CREDIT_ACTIVE', lambda x: (x == 'Active').sum()),  # 当前未结清活跃信贷数量
    bureau_closed_count    = ('CREDIT_ACTIVE', lambda x: (x == 'Closed').sum()),   # 已结清关闭信贷数量

    # ========== 2. 信贷金额负债类特征 ==========
    bureau_debt_sum        = ('AMT_CREDIT_SUM_DEBT', 'sum'),          # 客户当前总负债余额
    bureau_credit_sum      = ('AMT_CREDIT_SUM', 'sum'),              # 客户历史总授信额度
    bureau_overdue_sum     = ('AMT_CREDIT_SUM_OVERDUE', 'sum'),       # 客户当前全部逾期欠款总额

    # ========== 3. 信贷期限、最大逾期金额特征 ==========
    bureau_max_overdue     = ('AMT_CREDIT_MAX_OVERDUE', 'max'),       # 单条信贷历史最大逾期金额
    bureau_days_credit_mean= ('DAYS_CREDIT', 'mean'),                 # 所有信贷开户距今平均天数
    bureau_days_enddate_max= ('DAYS_CREDIT_ENDDATE', 'max'),          # 最近一笔信贷到期日距今天数

    # ========== 4. 从月度表聚合来的逾期历史指标（全局客户维度） ==========
    bureau_bb_max_dpd      = ('bb_max_dpd', 'max'),                   # 全账户历史最严重逾期等级
    bureau_bb_mean_dpd     = ('bb_mean_dpd', 'mean'),                 # 所有账户平均逾期严重程度
    bureau_bb_dpd_months_sum=('bb_dpd_months', 'sum'),                # 客户全部信贷累计发生逾期的总月数
).reset_index()

print(bureau_agg.shape)
bureau_agg.head()

(305811, 13)


,SK_ID_CURR,bureau_loan_count,bureau_active_count,bureau_closed_count,bureau_debt_sum,bureau_credit_sum,bureau_overdue_sum,bureau_max_overdue,bureau_days_credit_mean,bureau_days_enddate_max,bureau_bb_max_dpd,bureau_bb_mean_dpd,bureau_bb_dpd_months_sum
0,100001,7,3,4,596686.5,1453365.000,0.0,NaN,-735.000000,1778.0,1.0,0.007519,1.0
1,100002,8,2,6,245781.0,865055.565,0.0,5043.645,-874.000000,780.0,1.0,0.255682,27.0
2,100003,4,1,3,0.0,1017400.500,0.0,0.000,-1400.750000,1216.0,NaN,NaN,0.0
3,100004,2,0,2,0.0,189037.800,0.0,0.000,-867.000000,-382.0,NaN,NaN,0.0
4,100005,3,2,1,568408.5,657126.000,0.0,0.000,-190.666667,1324.0,0.0,0.000000,0.0


In [4]:
# 选取历史贷款表prev里5个时间天数字段，批量输出统计描述（均值、分位数、极值等）
# 字段含义：均为距离当前申请日的天数，负数=过去，正数=未来
prev[['DAYS_FIRST_DRAWING', 'DAYS_FIRST_DUE', 'DAYS_LAST_DUE_1ST_VERSION', 'DAYS_LAST_DUE', 'DAYS_TERMINATION']].describe()

,DAYS_FIRST_DRAWING,DAYS_FIRST_DUE,DAYS_LAST_DUE_1ST_VERSION,DAYS_LAST_DUE,DAYS_TERMINATION
count,997149.000000,997149.000000,997149.000000,997149.000000,997149.000000
mean,342209.855039,13826.269337,33767.774054,76582.403064,81992.343838
std,88916.115833,72444.869708,106857.034789,149647.415123,153303.516729
min,-2922.000000,-2892.000000,-2801.000000,-2889.000000,-2874.000000
25%,365243.000000,-1628.000000,-1242.000000,-1314.000000,-1270.000000
50%,365243.000000,-831.000000,-361.000000,-537.000000,-499.000000
75%,365243.000000,-411.000000,129.000000,-74.000000,-44.000000
max,365243.000000,365243.000000,365243.000000,365243.000000,365243.000000


In [5]:
# ========== 1、清洗异常填充值：统一把365243无效极值替换为空值NaN ==========
# 上一步describe发现5个时间字段存在统一填充脏值365243，建模无意义，替换成缺失值方便后续填充
prev['DAYS_FIRST_DRAWING'].replace(365243, np.nan, inplace=True)
prev['DAYS_FIRST_DUE'].replace(365243, np.nan, inplace=True)
prev['DAYS_LAST_DUE_1ST_VERSION'].replace(365243, np.nan, inplace=True)
prev['DAYS_LAST_DUE'].replace(365243, np.nan, inplace=True)
prev['DAYS_TERMINATION'].replace(365243, np.nan, inplace=True)

0           -37.0
1             NaN
2             NaN
3          -177.0
4             NaN
            ...  
1670209    -351.0
1670210   -1297.0
1670211   -1181.0
1670212    -817.0
1670213    -423.0
Name: DAYS_TERMINATION, Length: 1670214, dtype: float64

In [6]:
# ========== 2、衍生历史贷款信贷使用率特征 ==========
# 历史申请放款金额 / 历史授信总额，+1防止分母为0除零报错
prev['PREV_CREDIT_UTIL'] = prev['AMT_APPLICATION'] / (prev['AMT_CREDIT'] + 1)

# ========== 3、按客户ID分组，聚合所有历史贷款记录，生成客户级特征 ==========
prev_agg = prev.groupby('SK_ID_CURR').agg(
    # 历史申请总次数
    prev_app_count          = ('SK_ID_PREV', 'count'),
    # 历史申请中审批通过的笔数
    prev_approved_count     = ('NAME_CONTRACT_STATUS', lambda x: (x == 'Approved').sum()),
    # 历史申请中被拒绝的笔数
    prev_refused_count      = ('NAME_CONTRACT_STATUS', lambda x: (x == 'Refused').sum()),
    # 历史每笔贷款平均授信额度
    prev_amt_credit_mean    = ('AMT_CREDIT', 'mean'),
    # 历史每笔贷款平均月供
    prev_amt_annuity_mean   = ('AMT_ANNUITY', 'mean'),
    # 历史每笔贷款平均首付金额
    prev_amt_downpayment_mean = ('AMT_DOWN_PAYMENT', 'mean'),
    # 历史信贷使用率平均值
    prev_credit_util_mean   = ('PREV_CREDIT_UTIL', 'mean'),
    # 距离本次申请最近一笔历史贷款的决策天数（min负数代表过去时间最近）
    prev_days_decision_min  = ('DAYS_DECISION', 'min'),
).reset_index()

# ========== 4、衍生客户历史贷款审批通过率 ==========
# 通过笔数 / 总申请笔数，衡量客户过往信贷申请的资质好坏
prev_agg['prev_approval_rate'] = (
    prev_agg['prev_approved_count'] / prev_agg['prev_app_count']
)

print(prev_agg.shape)
prev_agg.head()

(338857, 10)


,SK_ID_CURR,prev_app_count,prev_approved_count,prev_refused_count,prev_amt_credit_mean,prev_amt_annuity_mean,prev_amt_downpayment_mean,prev_credit_util_mean,prev_days_decision_min,prev_approval_rate
0,100001,1,1,0,23787.00,3951.000,2520.0,1.044035,-1740,1.0
1,100002,1,1,0,179055.00,9251.775,0.0,0.999994,-606,1.0
2,100003,3,3,0,484191.00,56553.990,3442.5,0.949323,-2341,1.0
3,100004,1,1,0,20106.00,5357.250,4860.0,1.207639,-815,1.0
4,100005,2,1,0,20076.75,4813.200,4464.0,0.555573,-757,0.5


In [7]:
# 还款行为特征加工：判断客户每期是否按时还款、拖欠天数、拖欠金额
# 1. 衍生逾期天数：实际扣款日 - 应还款日
# 结果>0：逾期；=0：准时；<0：提前还款
inst['DAYS_LATE'] = inst['DAYS_ENTRY_PAYMENT'] - inst['DAYS_INSTALMENT']

# 2. 衍生少还/拖欠金额：当期应还月供 - 实际到账还款
# 结果>0：少还钱、有欠款；≤0：足额/超额还款
inst['AMT_UNDERPAID'] = inst['AMT_INSTALMENT'] - inst['AMT_PAYMENT']

# 按客户唯一ID分组，聚合该客户所有历史分期还款记录，生成客户级逾期特征
inst_agg = inst.groupby('SK_ID_CURR').agg(
    inst_count                = ('SK_ID_PREV', 'count'),          # 客户历史总分期还款期数
    inst_days_late_mean       = ('DAYS_LATE', 'mean'),           # 平均每期逾期天数（正数越多逾期越严重）
    inst_days_late_max        = ('DAYS_LATE', 'max'),            # 单期最大逾期天数（风控核心硬指标）
    inst_late_payments_count  = ('DAYS_LATE', lambda x: (x > 0).sum()), # 发生逾期的总期数
    inst_amt_underpaid_mean   = ('AMT_UNDERPAID', 'mean'),       # 平均每期拖欠金额
    inst_amt_underpaid_sum    = ('AMT_UNDERPAID', 'sum'),        # 历史累计拖欠总金额
).reset_index()

# 衍生客户历史逾期率：逾期期数 / 全部还款期数
# 直观衡量客户过往还款稳定性，逾期率越高坏账风险越大
inst_agg['inst_late_payment_rate'] = (
    inst_agg['inst_late_payments_count'] / inst_agg['inst_count']
)

print(inst_agg.shape)
inst_agg.head()

(339587, 8)


,SK_ID_CURR,inst_count,inst_days_late_mean,inst_days_late_max,inst_late_payments_count,inst_amt_underpaid_mean,inst_amt_underpaid_sum,inst_late_payment_rate
0,100001,7,-7.285714,11.0,1,0.0,0.0,0.142857
1,100002,19,-20.421053,-12.0,0,0.0,0.0,0.000000
2,100003,25,-7.160000,-1.0,0,0.0,0.0,0.000000
3,100004,3,-7.666667,-3.0,0,0.0,0.0,0.000000
4,100005,9,-23.555556,1.0,1,0.0,0.0,0.111111


In [8]:
# 信用卡表cc特征衍生
# 1. 信用卡使用率 = 当前欠款余额 / 实际授信额度，+1避免分母为0引发除零错误
cc['CC_UTIL'] = cc['AMT_BALANCE'] / (cc['AMT_CREDIT_LIMIT_ACTUAL'] + 1)

# 2. 汇总各类取现金额：ATM取现+普通取现+其他渠道取现+POS取现，得到总取现金额
cc['AMT_DRAWINGS_TOTAL'] = cc['AMT_DRAWINGS_ATM_CURRENT'] + cc['AMT_DRAWINGS_CURRENT'] + cc['AMT_DRAWINGS_OTHER_CURRENT'] + cc['AMT_DRAWINGS_POS_CURRENT']

# 3. 直接使用原始当期总还款额字段，避免与AMT_PAYMENT_CURRENT重复累加

# 按客户ID分组，将多条信用卡月度记录聚合为客户维度特征
cc_agg = cc.groupby('SK_ID_CURR').agg(
    cc_count            = ('SK_ID_PREV', 'count'),        # 客户持有的信用卡账户数量
    cc_balance_mean     = ('AMT_BALANCE', 'mean'),        # 信用卡平均欠款余额
    cc_balance_max      = ('AMT_BALANCE', 'max'),         # 单张信用卡最高欠款余额
    cc_drawings_mean    = ('AMT_DRAWINGS_TOTAL', 'mean'), # 平均每月取现金额
    cc_payment_rate_mean= ('AMT_PAYMENT_TOTAL_CURRENT', 'mean'), # 平均每月还款金额
    cc_util_mean        = ('CC_UTIL', 'mean'),            # 信用卡平均使用率（强风险特征）
    cc_util_max         = ('CC_UTIL', 'max'),             # 单张信用卡最高使用率
    cc_dpd_max          = ('SK_DPD', 'max'),              # 信用卡历史最长逾期天数DPD
    cc_dpd_mean         = ('SK_DPD', 'mean'),             # 信用卡平均逾期天数
).reset_index()

print(cc_agg.shape)
cc_agg.head()

(103558, 10)


,SK_ID_CURR,cc_count,cc_balance_mean,cc_balance_max,cc_drawings_mean,cc_payment_rate_mean,cc_util_mean,cc_util_max,cc_dpd_max,cc_dpd_mean
0,100006,6,0.000000,0.00,NaN,0.000000,0.000000,0.000000,0,0.000000
1,100011,74,54482.111149,189000.00,4864.864865,4520.067568,0.302677,1.049994,0,0.000000
2,100013,96,18159.919219,161420.22,12700.000000,6817.172344,0.115300,1.024884,1,0.010417
3,100021,17,0.000000,0.00,NaN,0.000000,0.000000,0.000000,0,0.000000
4,100023,8,0.000000,0.00,NaN,0.000000,0.000000,0.000000,0,0.000000


In [9]:
#POS表（消费分期/小额商品分期月度台账），按客户SK_ID_CURR聚合生成客户维度特征
pos_agg = pos.groupby('SK_ID_CURR').agg(
    pos_count                = ('SK_ID_PREV', 'count'),                 # 客户名下POS分期总账户数量
    pos_months_balance       = ('MONTHS_BALANCE', 'mean'),              # 分期记录平均账龄月份
    pos_cnt_instalment       = ('CNT_INSTALMENT', 'mean'),              # 每笔分期平均总期数
    pos_dpd_max              = ('SK_DPD', 'max'),                       # POS分期历史最长逾期天数DPD（风控重点指标）
    pos_dpd_mean             = ('SK_DPD', 'mean'),                      # POS分期平均逾期天数
    pos_dpd_nonzero          = ('SK_DPD', lambda x: (x > 0).sum()),     # 发生逾期的月度记录条数
    pos_completed_count      = ('NAME_CONTRACT_STATUS',
                                lambda x: (x == 'Completed').sum()),    # 正常结清的分期合同数量
).reset_index()

print(pos_agg.shape)
pos_agg.head()

(337252, 8)


,SK_ID_CURR,pos_count,pos_months_balance,pos_cnt_instalment,pos_dpd_max,pos_dpd_mean,pos_dpd_nonzero,pos_completed_count
0,100001,9,-72.555556,4.000000,7,0.777778,1,2
1,100002,19,-10.000000,24.000000,0,0.000000,0,0
2,100003,28,-43.785714,10.107143,0,0.000000,0,2
3,100004,4,-25.500000,3.750000,0,0.000000,0,1
4,100005,11,-20.000000,11.700000,0,0.000000,0,1


In [10]:
master = app_train.copy()

for agg_df, name in zip(
    [bureau_agg, prev_agg, inst_agg, cc_agg, pos_agg],
    ['bureau', 'prev', 'inst', 'cc', 'pos']
):
    before = master.shape[1]
    # how='left'：保留主表全部客户，没有对应历史数据的客户填充NaN
    master = master.merge(agg_df, on='SK_ID_CURR', how='left')
    after = master.shape[1]
    print(f"After joining {name}: {master.shape} (+{after - before} features)")

print(f"\nFinal master table shape: {master.shape}")

After joining bureau: (307511, 134) (+12 features)
After joining prev: (307511, 143) (+9 features)
After joining inst: (307511, 150) (+7 features)
After joining cc: (307511, 159) (+9 features)
After joining pos: (307511, 166) (+7 features)

Final master table shape: (307511, 166)


In [11]:
display(master['TARGET'].value_counts())

# 注释含义：NaN代表该客户没有对应历史借贷记录
new_missing = master.isnull().mean() * 100
new_missing = new_missing[new_missing > 0].sort_values(ascending=False)
print(f"\nFeatures with missing values: {len(new_missing)}")
print(new_missing.head(20))

import os
os.makedirs('../data/processed', exist_ok=True)
# parquet优势：压缩、读写速度快，适合大数据建模
master.to_parquet('../data/processed/master_table.parquet', engine='fastparquet', index=False)

print(f"Final shape: {master.shape}")

TARGET
0    282686
1     24825
Name: count, dtype: int64


Features with missing values: 111
cc_drawings_mean            80.117784
cc_util_mean                71.739222
cc_dpd_max                  71.739222
cc_count                    71.739222
cc_balance_mean             71.739222
cc_payment_rate_mean        71.739222
cc_util_max                 71.739222
cc_balance_max              71.739222
cc_dpd_mean                 71.739222
bureau_bb_mean_dpd          70.007252
bureau_bb_max_dpd           70.007252
COMMONAREA_MEDI             69.872297
COMMONAREA_AVG              69.872297
COMMONAREA_MODE             69.872297
NONLIVINGAPARTMENTS_AVG     69.432963
NONLIVINGAPARTMENTS_MEDI    69.432963
NONLIVINGAPARTMENTS_MODE    69.432963
FONDKAPREMONT_MODE          68.386172
LIVINGAPARTMENTS_MEDI       68.354953
LIVINGAPARTMENTS_MODE       68.354953
dtype: float64


Final shape: (307511, 166)


## Feature Engineering Summary

- **Source tables joined:** 7 (bureau, bureau_balance, previous_application,
  installments_payments, credit_card_balance, POS_CASH_balance)
- **Final master table:** (307511, 166)
- **Key engineered features:**
  - `bureau_bb_max_dpd` — worst ever delinquency in credit bureau history
  - `inst_late_payment_rate` — share of late installment payments
  - `prev_approval_rate` — ratio of approved to total previous applications
  - `cc_util_max` — peak credit card utilization
  - `pos_dpd_nonzero` — number of months with any DPD in POS loans
- **NaN after join** = applicant has no history in that table — will be treated
  as a separate category in WoE binning (notebook 04)

**Next step:** WoE/IV feature selection → notebook 04 

## Point-in-time Feature Availability
Historical aggregates from bureau, previous applications, installments, credit card and POS/CASH tables must use only records available at or before the application decision time.
The public dataset lacks a reliable application timestamp; this notebook therefore feeds a stratified random holdout rather than a true temporal OOT sample.
